In [1]:
from math import exp,tanh
import random
import numpy as np

In [2]:
class Value:
    def __init__(self,data,children=(),op='',label=''):
        self.data=data
        self._prev=set(children)
        self._op=op
        self.grad=0
        self._backward=lambda:None
        self.label=label
    def __repr__(self):
        return f'Value=({self.data})'
    def __add__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data+other.data,(self,other),op='+')
        
        def _backward():
            self.grad+=1*out.grad
            other.grad+=1*out.grad
        out._backward=_backward
        return out
        
   
    def __mul__(self,other):
        other=other if isinstance(other,Value) else Value(other)
        out=Value(self.data*other.data,(self,other),'*')
        def _backward():
            self.grad+=other.data*out.grad
            other.grad+=self.data*out.grad
        out._backward=_backward
        return out
    def __rmul__(self,other):
        return self*other
    def __radd__(self,other):
        return self+other
    def __neg__(self):
        return self*-1
    def __sub__(self,other):
        out=self+(-other)
        return out
    def relu(self):
        out=Value(max(self.data,0),(self,),'relu')
        def _backward():
            self.grad+=(out.data>0)*out.grad
        out._backward=_backward
        return out
    def __pow__(self,other):
        assert isinstance(other, (int, float)), "Only supporting int/float powers"
        out=Value(self.data**other,(self,),f'**{other}')
        def _backward():
            self.grad+=other*self.data**(other-1)*out.grad
        out._backward=_backward
        return out
    def __truediv__(self,other):
        out=self*other**-1
        return out
    def exp(self):
        out=Value(exp(self.data),(self,),'exp')
        def _backward():
            self.grad+=out.data*out.grad
        out._backward=_backward
        return out
    def tanh(self):
        t=tanh(self.data)
        out=Value(t,(self,),'tanh')
        def _backward():
            self.grad+=(1-t**2)*out.grad
        out._backward=_backward
        return out
    def backward(self):
        topo=[]
        visited=set()
        def build_topo(v):
            if v  not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad=1
        for node in reversed(topo):
            node._backward()
            
        

In [3]:
a=Value(3)
b=Value(2)
c=a+b
e=c.exp()
o=(e-1)/(e+1)


In [4]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [176]:
class Neuron:
    def __init__(self,nin):
        self.w=[Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b=Value(random.uniform(-1,1))
    def __call__(self,x):
        act=sum([xi*wi for xi,wi in zip(x,self.w)],self.b)
        
        return act.tanh()
    def parameters(self):
        return self.w+[self.b]
class Layer:
    def __init__(self,nin,nout):
        self.neurons=[Neuron(nin) for _ in range(nout)]
    def __call__(self,x):
        out=[n(x) for n in self.neurons]
        return out[0] if len(out)==1 else out
    def parameters(self):
        para=[p for neuron in self.neurons for p in neuron.parameters()]
        return para
        
class MLP:
    
    def __init__(self,nin,nouts):
        sz=[nin]+nouts
        self.layers=[Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    def __call__(self,x):
        for layer in self.layers:
            x=layer(x)
        return x
    def parameters(self):
        para=[p for layer in self.layers for p in layer.parameters()]
        return para

In [177]:
xs=[[2.0,3.0,-1.0],
    [3.0,-1,0.5],
    [0.5,1.0,1.0],
   [1.0,1.0,-1.0]]
    
n=MLP(3,[4,4,1])

In [178]:
y_pred=[n(x) for x in xs]
ys=[1,1,-1,-1]
y_pred

[Value=(-0.9069201031066632),
 Value=(-0.7243097113432135),
 Value=(-0.7839125011132475),
 Value=(-0.9075824755149664)]

In [179]:
loss=sum((yout-ygt)**2 for ygt,yout in zip(ys,y_pred))
loss

Value=(6.664823066271917)

In [201]:
for k in range(20):
    y_pred=[n(x) for x in xs]
    loss=sum((yout-ygt)**2 for ygt,yout in zip(ys,y_pred))
    for p in n.parameters():
        p.grad=0
    loss.backward()
    for p in n.parameters():
        p.data+=-0.05*p.grad
    print(k,loss.data)
    

0 0.010415305756012731
1 0.010315018719866287
2 0.010216535891015678
3 0.010119810314660556
4 0.010024796623817668
5 0.00993145097358454
6 0.009839730978611124
7 0.00974959565359965
8 0.009661005356664566
9 0.009573921735394333
10 0.009488307675467319
11 0.009404127251682555
12 0.009321345681275274
13 0.009239929279394614
14 0.009159845416628397
15 0.009081062478466817
16 0.0090035498266034
17 0.00892727776197745
18 0.008852217489467993
19 0.008778341084154179


In [202]:
y_pred=[n(x) for x in xs]

In [203]:
y_pred

[Value=(0.9451220305850253),
 Value=(0.9513896339851688),
 Value=(-0.9692551772327366),
 Value=(-0.9511551628023465)]